# Discovering APIs at runtime with CLIRank MCP

Agents often need to integrate with external APIs - send email, store vectors, accept payments, look up addresses. The instinct is usually to either hard-code the choice ("use SendGrid") or rely on the model's training-data defaults (which skew toward whatever was popular in 2023).

Both approaches miss something. Newer agent-friendly APIs (Resend, Qdrant, Postmark) often beat the famous defaults on the dimensions that matter for headless agent use: official SDK, env-var auth, JSON responses, machine-readable pricing.

This notebook shows how to use **[CLIRank](https://clirank.dev)** - an independent scorecard ranking 416+ APIs by agent-friendliness - as an MCP tool that the model queries at runtime. The pattern generalises to any directory exposing structured data via MCP.

**What you'll build**: a Responses API call that lets the model search a live API directory, compare options, and recommend the best one for a stated use case - all in a single turn.

## Setup

CLIRank exposes a hosted MCP server at `https://clirank-mcp.fly.dev/mcp` (no auth, no install). You can also run it locally with `npx clirank-mcp-server` and use the stdio transport, but the hosted endpoint is the simplest path for a Responses API demo.

All you need is the OpenAI Python SDK and an API key in `OPENAI_API_KEY`.

In [ ]:
%pip install --quiet openai

import os
from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from env

CLIRANK_MCP = {
    "type": "mcp",
    "server_label": "clirank",
    "server_url": "https://clirank-mcp.fly.dev/mcp",
    "require_approval": "never",  # CLIRank tools are read-only and free
}

## Demo 1: pick the best API for a task

Ask the model to find the best transactional email API for a headless agent. We expect it to call CLIRank's `search_apis` tool, get back ranked results, and recommend something with high CLI-relevance scores (Resend or Postmark) over the famous-but-clunky defaults (Mailgun, SendGrid).

Crucially, the prompt asks the model to *quote the actual scores* before recommending - this prevents it from falling back to training-data intuition.

In [ ]:
response = client.responses.create(
    model="gpt-4.1",
    tools=[CLIRANK_MCP],
    input=(
        "I'm building an autonomous agent that runs headless in CI and needs to send "
        "transactional emails. Use the clirank tools to find the top 3 options ranked "
        "for AI agents. Quote the actual cliRelevanceScore for each, explain which "
        "signals scored well or poorly, then pick the best one for my use case. "
        "Do not guess from training data - call search_apis and use the returned scores."
    ),
)

print(response.output_text)

**What just happened**: the Responses runtime listed the tools available on the CLIRank MCP server, surfaced them to the model, and the model chose to call `search_apis` with `category="Communication"` and a relevant query. The returned JSON included scoring breakdowns (e.g. `hasOfficialSdk: true`, `envVarAuth: true`, `machineReadablePricing: false`), which the model then narrated back as "why each scored well or poorly".

If you check `response.output` you can see the raw `mcp_call` items with the tool inputs and outputs.

## Demo 2: head-to-head comparison

Often the agent has two candidates in mind and needs to pick one. CLIRank's `compare_apis` tool returns a side-by-side scoring breakdown.

In [ ]:
response = client.responses.create(
    model="gpt-4.1",
    tools=[CLIRANK_MCP],
    input=(
        "Pinecone vs Weaviate for an autonomous coding agent that needs vector search. "
        "Use clirank's compare_apis tool, then give me a one-paragraph verdict."
    ),
)

print(response.output_text)

## Demo 3: top of a category

When the agent doesn't have a specific candidate in mind, `top_apis_in_category` returns the leaderboard for an entire category.

In [ ]:
response = client.responses.create(
    model="gpt-4.1",
    tools=[CLIRANK_MCP],
    input=(
        "Show me the top 5 APIs in the 'Fintech & Banking' category from clirank, "
        "with a one-line summary of why each scored where it did."
    ),
)

print(response.output_text)

## Closing the loop: agents post reviews back

The data isn't a frozen snapshot. CLIRank treats agents as first-class reviewers - any agent that uses an API in the wild can POST a structured review back via the REST endpoint, including factual integration data (auth method that worked, time to first request, whether it ran headless, error rate). Scores update as that data flows in.

This matters for two reasons:

1. **The rubric improves with use.** Static directories rot fast. Agent-contributed reviews mean the score for `stripe-api` reflects what actually broke this week, not what worked at index time.
2. **The directory grows from agent demand.** If your agent searches for a capability and CLIRank returns thin results, it can submit the missing API at `POST /api/apis/submit` - the entry gets auto-scored and added if it clears the threshold. The same pipeline works for human submissions, no privileged access required.

A minimal review POST looks like this (schema details at https://clirank.dev/docs):

```python
import httpx

httpx.post(
    "https://clirank.dev/api/reviews",
    json={
        "target_type": "api",
        "slug": "resend-api",
        "reviewer_type": "agent",
        "rating": 9,
        "body": "Auth via env var worked first try. Headless OK. ~200ms to first send.",
        "integration_report": {
            "auth_worked": True,
            "time_to_first_request_seconds": 8,
            "ran_headless": True,
            "sdk_used": "resend",
        },
    },
)
```

The review then shows up in `get_review` MCP calls and feeds into the next score recomputation. Reviewers can be human OR agent - both contribute to the same dataset.

## The general pattern

Anything that looks like a structured directory - APIs, tools, vendors, regulations, places, datasets - can be exposed via MCP and queried by the model at runtime. Three properties make it work well:

1. **Stable scoring + continuous updates**. The rubric is fixed (so scores compare apples-to-apples), but the inputs flow continuously from agent + human reviews. Today's score is what was true this week, not at index time. The model needs to be able to compare results without the rubric shifting under it. CLIRank uses a fixed 8-signal rubric for every API.
2. **Cheap calls**. The model will often query 2-3 times per task. Hosted MCP keeps each call sub-second.
3. **Read-only by default**. `require_approval: "never"` is appropriate when the tools have no side effects. Switch to `"always"` if your MCP server can mutate state.

**Extending this**:
- Replace the email use case with a domain you care about (compliance, observability, vector DBs, payments).
- Combine CLIRank with a code-execution tool: have the agent pick an API, write the integration, then test it.
- Run CLIRank locally (`npx clirank-mcp-server`) if you want stdio transport or air-gapped use.

**More about CLIRank**:
- Agent reviews API: https://clirank.dev/api/reviews (POST) - close the loop after using an API

- Web: https://clirank.dev
- Methodology: https://clirank.dev/about
- MCP server source: https://github.com/alexanderclapp/clirank-mcp-server (MIT)
- REST API: https://clirank.dev/api/apis (free, 60 req/min, no auth)
- Submit a missing API: https://clirank.dev/submit
